# 00. Homework description

In the seminar, you implemented a VQ-VAE for MNIST. In this homework, you will transfer the same ideas to speech.

This homework uses [LibriSpeech](https://www.openslr.org/12): short 16 kHz speech segments for training.

A small 1D convolutional autoencoder is already provided in `model.py`. Your job is to implement the quantization components in `quantizers.py`, run the experiments, and interpret the results.

Scoring:
- Part 1: baseline — 0.5 points
- Part 2: naive VQ — 1.5 points
- Part 3: EMA + restart — 2.5 points
- Part 4: FSQ — 2.5 points
- Part 5: RVQ + progressive decoding — 3 points

Total: **10 points**.

The homework submission should include:
- Completed notebook
- Answers to questions
- Attached loss graphs from TensorBoard
- Reconstructed audio files

In [ ]:
from pathlib import Path
import torch
import lightning as L
from IPython.display import Audio, display
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from data import LibriSpeechDataModule
from model import AudioAutoEncoder, MultiResolutionSTFTLoss
from quantizers import (
    VectorQuantizer, VectorQuantizerEMA, VectorQuantizerRestart,
    FiniteScalarQuantizer, ResidualVectorQuantizer,
    VectorQuantizationLoss, CommitmentLoss,
)

In [ ]:
seed = 42
L.seed_everything(seed, workers=True)

device_id = 0
device = torch.device(f"cuda:{device_id}" if device_id >= 0 else "cpu")

data_path = Path("./data")
data_path.mkdir(exist_ok=True)

batch_size = 128
datamodule = LibriSpeechDataModule(data_dir=str(data_path), batch_size=batch_size)

In [ ]:
def make_trainer(name: str, max_epochs: int = 30):
    run_dir = data_path / "runs" / name
    run_dir.mkdir(parents=True, exist_ok=True)
    return L.Trainer(
        callbacks=[
            EarlyStopping(monitor="val/loss", patience=5),
            ModelCheckpoint(dirpath=run_dir / "ckpt", save_top_k=1, monitor="val/loss"),
        ],
        devices=[device_id] if device_id >= 0 else "auto",
        accelerator="gpu" if device_id >= 0 else "cpu",
        max_epochs=max_epochs,
        log_every_n_steps=10,
        num_sanity_val_steps=0,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        default_root_dir=str(run_dir),
    )

---
## 01. Baseline Autoencoder (0.5 points)

Before introducing quantization, train the plain autoencoder with the default waveform-L1 reconstruction loss.

If you want, you can later try adding a multi-resolution STFT term and compare the result, but training may be less stable in this setup.

In [ ]:
trainer_l1 = make_trainer("part1_l1")
model_l1 = AudioAutoEncoder(spectral_weight=0.0, waveform_weight=1.0)
trainer_l1.fit(model_l1, datamodule=datamodule)

In [ ]:
# Optional: later you can try adding an STFT term and compare the result.
# In this setup it may make training less stable.

In [ ]:
n_params = sum(p.numel() for p in model_l1.parameters())
print(f"Backbone: {n_params:,} parameters  (~{n_params/1e6:.1f} M)")

In [ ]:
def make_quantized_model(quantizer, vq_loss, lr: float = 1e-4):
    return AudioAutoEncoder(
        quantizer=quantizer,
        vq_loss=vq_loss,
        lr=lr,
    )

### Task (0.5 points)

Evaluate the trained model and add reconstruction examples. Is it strong enough to serve as a reference upper bound for the later quantized models?


In [ ]:
def display_audio_reconstructions(model, datamodule, num_examples: int = 2, offset: int = 42):
    model.prepare_for_inference().to(device)
    sample_audios, example_ids = datamodule.sample_val_examples(num_examples=num_examples, offset=offset)
    sample_audios = sample_audios.to(device)

    # <YOUR CODE HERE>

---
## 02. Naive VQ and Codebook Collapse (1.5 points)

Add a codebook via nearest-neighbour quantization. 

All quantized models in this homework use the same architecture and optimizer settings. The naive VQ run is still expected to be less stable than the later variants, so part of the exercise is to observe what goes wrong.


### Task (1 point)

Open `quantizers.py` and implement `VectorQuantizer.encode` and `.decode`.

In [ ]:
trainer_vq = make_trainer("part2_naive_vq")
model_vq = make_quantized_model(
    quantizer=VectorQuantizer(codebook_size=256, embedding_dim=128),
    vq_loss=VectorQuantizationLoss(commitment_cost=0.25),
    lr=1e-4,
)
trainer_vq.fit(model_vq, datamodule=datamodule)

In [ ]:
datamodule.setup("fit")
usage_vq = model_vq.compute_codebook_usage(datamodule.val_dataloader())
print(usage_vq["perplexity"])
fig_vq = model_vq.plot_codebook_usage(usage_vq)

### Task (0.5 points)

Inspect codebook usage at different epochs. How does the distribution evolve? Why does naive VQ tend to collapse?

---
## 03. Preventing Codebook Collapse (2.5 points)

Previously the main failure mode for trained models was not poor optimization in general, but poor utilization of the codebook. In this part, you will test two common fixes.

### 03a. EMA codebook updates (1 point)

Rather than updating the codebook through gradients, maintain exponential moving averages of:

- how many times each codebook entry is selected
- the sum of encoder outputs assigned to each entry

In `VectorQuantizerEMA.forward`, first quantize z as usual. Then update `ema_counts` and `ema_embedding_sum` using the current batch assignments and the EMA decay. After that, update each codebook vector as
`codebook[i] = ema_embedding_sum[i] / (ema_count[i] + eps).`

The codebook should not receive gradients. When training with this quantizer, use `CommitmentLoss` rather than `VectorQuantizationLoss`.

Implement `VectorQuantizerEMA.forward`.


In [ ]:
trainer_ema = make_trainer("part3a_ema")
model_ema = make_quantized_model(
    quantizer=VectorQuantizerEMA(codebook_size=256, embedding_dim=128, decay=0.99),
    vq_loss=CommitmentLoss(commitment_cost=0.25),
    lr=1e-4,
)
trainer_ema.fit(model_ema, datamodule=datamodule)

In [ ]:
usage_ema = model_ema.compute_codebook_usage(datamodule.val_dataloader())
print(usage_ema["perplexity"])
fig_ema = model_ema.plot_codebook_usage(usage_ema)

### 03b. Random restart for dead entries (1 point)

Track how many consecutive forward passes each codebook entry has gone without being assigned. If an entry has not been selected in `threshold` steps, reinitialize it to a random encoder output from the current batch. This prevents entries from drifting permanently out of the encoder's distribution.

Implement `VectorQuantizerRestart._maybe_restart`.

In [ ]:
trainer_restart = make_trainer("part3b_restart")
model_restart = make_quantized_model(
    quantizer=VectorQuantizerRestart(codebook_size=256, embedding_dim=128, threshold=20),
    vq_loss=VectorQuantizationLoss(commitment_cost=0.25),
    lr=1e-4,
)
trainer_restart.fit(model_restart, datamodule=datamodule)

In [ ]:
usage_restart = model_restart.compute_codebook_usage(datamodule.val_dataloader())
print(usage_restart["perplexity"])
fig_restart = model_restart.plot_codebook_usage(usage_restart)

### Task (0.5 points)

Compare EMA and random restart in terms of codebook utilization and reconstruction quality. If the rankings differ, why can that still happen?


---
## 04. Finite Scalar Quantization (2.5 points)

Finite Scalar Quantization [(Mentzer et al., 2023)](https://arxiv.org/abs/2309.15505) takes a structurally different approach to the quantization problem. Rather than maintaining a learned codebook and performing nearest-neighbour lookup, each latent dimension is independently rounded to a small fixed set of integer values. For example, with 5 levels per dimension, the valid codes are $\{-2, -1, 0, 1, 2\}$.

With 4 dimensions at 5 levels each the effective codebook size is $5^4 = 625$, with no entries to learn and no collapse mechanism. Gradients flow through the rounding operation via a straight-through estimator, similar to standard VQ.

Because the quantization is applied independently per dimension, the latent must be low-dimensional for the scheme to work well. `FiniteScalarQuantizer` includes learned linear projections from `embedding_dim` down to `len(levels)` dimensions and back.

### Task (2 points)

Implement `FiniteScalarQuantizer._quantize`, `.encode`, and `.forward`. 

Compare FSQ's training dynamics and final reconstruction quality to the better of the VQ model.

In [ ]:
trainer_fsq = make_trainer("part4_fsq")
model_fsq = make_quantized_model(
    quantizer=FiniteScalarQuantizer(levels=[5, 5, 5, 5], embedding_dim=128),
    vq_loss=None,
    lr=1e-4,
)
trainer_fsq.fit(model_fsq, datamodule=datamodule)

### Task (0.5 points)

Obtain the integer codes, plot histograms of quantized values for each dimension over the validation set. Do you observe any pattern?

---
## 05. Residual Vector Quantization and Progressive Decoding (3 points)

Residual Vector Quantization stacks several quantizers. The first quantizer captures the coarse structure of the latent, and each later quantizer encodes the remaining residual.

After training the model can decode with different numbers of codebooks. Using only the first `k` codebooks gives a lower-bitrate reconstruction, using more codebooks gives a higher-quality reconstruction.


### Task (2 points)

Implement `ResidualVectorQuantizer.quantize_with_loss`, `.encode`, and `.decode`, then verify the progressive decoding idea experimentally.

In [ ]:
trainer_rvq = make_trainer("part5_rvq", max_epochs=40)
model_rvq = make_quantized_model(
    quantizer=ResidualVectorQuantizer(
        codebook_size=256,
        embedding_dim=128,
        n_codebooks=8,
        quantizer_cls=VectorQuantizer,
    ),
    vq_loss=VectorQuantizationLoss(commitment_cost=0.25),
    lr=1e-4,
)
trainer_rvq.fit(model_rvq, datamodule=datamodule)

### Progressive decoding (1 point)

After training the RVQ model, decode the validation set using only the first `k = 1, 2, 4, 8` codebooks.

For each value of `k`:
- compute the spectral reconstruction loss and bitrate
- log the same validation clips
- compare how quality changes as more codebooks are added

As `k` increases from 1 to 2 to 4 to 8, how do the reconstructions change? What is the smallest value of `k` that still gives acceptable speech quality?


In [ ]:
model_rvq.prepare_for_inference().to(device)
datamodule.setup("fit")
spec_loss_fn = MultiResolutionSTFTLoss().to(device)

# <YOUR CODE HERE>

In [ ]:
def display_progressive_decoding_examples(model, datamodule, n_layers_list=(1, 2, 4, 8), num_examples: int = 2, offset: int = 17):
    model.prepare_for_inference().to(device)
    sample_audios, example_ids = datamodule.sample_val_examples(num_examples=num_examples, offset=offset)
    sample_audios = sample_audios.to(device)

    # <YOUR CODE HERE>

In [ ]:
display_progressive_decoding_examples(model_rvq, datamodule, n_layers_list=(1, 2, 4, 8), num_examples=2, offset=42)